In [5]:
from pathlib import Path
import warnings

import gradio as gr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, shapiro
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm


# =========================================================
# 設定：CSV 預設與本程式放在同一資料夾
# =========================================================

APP_DIR = (
    Path(__file__).resolve().parent
    if "__file__" in globals()
    else Path.cwd()
)

MSA_FILE = APP_DIR / "msa_synthetic_data.csv"
PROCESS_FILE = APP_DIR / "process_synthetic_data.csv"

# X-bar / R 管制圖常數：此範例限每組 n=4
A2 = 0.729
D3 = 0.000
D4 = 2.282
d2 = 2.059

# 教學用門檻，不是通用驗收標準
MSA_MAX_PCT_TOLERANCE = 30.0
MSA_MIN_NDC = 5
MIN_SUBGROUPS = 20


# =========================================================
# 工具函式
# =========================================================

def empty_outputs(message):
    """發生錯誤時回傳一致數量的輸出，避免介面輸出數目不合。"""
    empty_df = pd.DataFrame()
    return (
        message,
        empty_df,  # MSA 原始資料
        empty_df,  # 製程原始資料
        empty_df,  # MSA 摘要
        empty_df,  # ANOVA
        empty_df,  # 子組摘要
        empty_df,  # 能力指標
        None,       # MSA 圖
        None,       # 管制圖
        None,       # 能力圖
    )


def check_csv_exists(path):
    if not path.exists():
        raise FileNotFoundError(
            f"找不到檔案：{path}\n"
            "請確認 CSV 檔案與 spc_gradio_app.py 位於同一資料夾。"
        )


def calculate_grr(msa, lsl, usl):
    """平衡交叉設計的教學用 ANOVA Gage R&R 計算。"""

    required = {"part", "operator", "repeat", "measurement"}
    missing = required - set(msa.columns)
    if missing:
        raise ValueError(f"MSA CSV 缺少必要欄位：{sorted(missing)}")

    data = msa.copy()
    data["measurement"] = pd.to_numeric(
        data["measurement"], errors="coerce"
    )

    if data[list(required)].isna().any().any():
        raise ValueError("MSA 資料的必要欄位有缺值或非數值資料。")

    if data.duplicated(["part", "operator", "repeat"]).any():
        raise ValueError("MSA 資料有重複的 part/operator/repeat 組合。")

    counts = data.groupby(["part", "operator"], observed=True).size()
    if counts.empty or counts.nunique() != 1:
        raise ValueError("MSA 資料不是平衡交叉設計，無法使用此範例算法。")

    n_parts = data["part"].nunique()
    n_operators = data["operator"].nunique()
    n_repeats = int(counts.iloc[0])

    if n_parts < 5:
        raise ValueError(f"MSA 零件數只有 {n_parts}，本範例至少需要 5 個。")
    if n_operators < 2:
        raise ValueError("MSA 至少需要 2 位作業員。")
    if n_repeats < 2:
        raise ValueError("每個零件與作業員組合至少需要 2 次重複量測。")

    model = smf.ols(
        "measurement ~ C(part) * C(operator)",
        data=data
    ).fit()

    anova = anova_lm(model, typ=2)

    # 不同 statsmodels 版本的欄位可能不同；
    # 若沒有 mean_sq，便由 sum_sq / df 計算。
    if "mean_sq" not in anova.columns:
        anova["mean_sq"] = anova["sum_sq"] / anova["df"]

    expected_terms = [
        "C(part)",
        "C(operator)",
        "C(part):C(operator)",
        "Residual",
    ]
    missing_terms = [term for term in expected_terms if term not in anova.index]
    if missing_terms:
        raise ValueError(
            f"ANOVA 表缺少預期項目：{missing_terms}\n"
            f"實際列名稱：{anova.index.tolist()}"
        )

    ms_part = anova.loc["C(part)", "mean_sq"]
    ms_operator = anova.loc["C(operator)", "mean_sq"]
    ms_interaction = anova.loc["C(part):C(operator)", "mean_sq"]
    ms_repeatability = anova.loc["Residual", "mean_sq"]

    # 平衡交叉 ANOVA 方法矩估計；負變異成分截為 0
    var_repeatability = max(ms_repeatability, 0.0)
    var_interaction = max(
        (ms_interaction - ms_repeatability) / n_repeats,
        0.0
    )
    var_operator = max(
        (ms_operator - ms_interaction) / (n_parts * n_repeats),
        0.0
    )
    var_part = max(
        (ms_part - ms_interaction) / (n_operators * n_repeats),
        0.0
    )

    var_grr = var_repeatability + var_operator + var_interaction
    var_total = var_grr + var_part

    sd_repeatability = np.sqrt(var_repeatability)
    sd_operator = np.sqrt(var_operator)
    sd_interaction = np.sqrt(var_interaction)
    sd_grr = np.sqrt(var_grr)
    sd_part = np.sqrt(var_part)
    sd_total = np.sqrt(var_total)

    tolerance = usl - lsl
    if tolerance <= 0:
        raise ValueError("規格錯誤：USL 必須大於 LSL。")

    pct_study = (
        100 * sd_grr / sd_total if sd_total > 0 else np.nan
    )
    pct_tolerance = 100 * 6 * sd_grr / tolerance
    ndc = (
        int(np.floor(1.41 * sd_part / sd_grr))
        if sd_grr > 0 else 0
    )

    msa_pass = (
        pct_tolerance <= MSA_MAX_PCT_TOLERANCE
        and ndc >= MSA_MIN_NDC
    )

    summary = pd.DataFrame([
        ["零件數", n_parts],
        ["作業員數", n_operators],
        ["每組重複量測數", n_repeats],
        ["重複性 SD", sd_repeatability],
        ["作業員變異 SD", sd_operator],
        ["Part × Operator SD", sd_interaction],
        ["總 Gage R&R SD", sd_grr],
        ["零件間 SD", sd_part],
        ["%Study Variation", pct_study],
        ["%Tolerance", pct_tolerance],
        ["ndc", ndc],
        [
            "教學用 MSA 閘門",
            "通過" if msa_pass else "未通過"
        ],
    ], columns=["指標", "結果"])

    return {
        "summary": summary,
        "anova": anova.reset_index(names="變異來源"),
        "msa_pass": msa_pass,
        "pct_study": pct_study,
        "pct_tolerance": pct_tolerance,
        "ndc": ndc,
    }


def make_msa_plot(msa):
    means = (
        msa.groupby(["part", "operator"], observed=True)["measurement"]
        .mean()
        .unstack("operator")
    )

    fig, ax = plt.subplots(figsize=(8, 4.5))

    for operator in means.columns:
        ax.plot(
            means.index.astype(str),
            means[operator],
            marker="o",
            label=f"Operator {operator}"
        )

    ax.set_title("MSA: Part × Operator Mean")
    ax.set_xlabel("Part")
    ax.set_ylabel("Mean Measurement")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    return fig


def analyze_xbar_r(process):
    required = {"subgroup", "measurement"}
    missing = required - set(process.columns)
    if missing:
        raise ValueError(f"製程 CSV 缺少必要欄位：{sorted(missing)}")

    data = process.copy()
    data["measurement"] = pd.to_numeric(
        data["measurement"], errors="coerce"
    )

    if data[list(required)].isna().any().any():
        raise ValueError("製程資料的 subgroup 或 measurement 有缺值或非數值。")

    group_sizes = data.groupby("subgroup", observed=True).size()

    if len(group_sizes) < MIN_SUBGROUPS:
        raise ValueError(
            f"子組數為 {len(group_sizes)}，少於範例設定的 {MIN_SUBGROUPS}。"
        )

    if group_sizes.nunique() != 1:
        raise ValueError("子組大小不一致；本範例不自動套用 X-bar/R 常數。")

    subgroup_size = int(group_sizes.iloc[0])
    if subgroup_size != 4:
        raise ValueError(
            f"本範例使用每組 n=4 的管制圖常數，"
            f"目前每組 n={subgroup_size}。"
        )

    summary = (
        data.groupby("subgroup", observed=True)["measurement"]
        .agg(
            xbar="mean",
            range=lambda x: x.max() - x.min()
        )
        .reset_index()
        .sort_values("subgroup")
    )

    xbarbar = summary["xbar"].mean()
    rbar = summary["range"].mean()

    xbar_ucl = xbarbar + A2 * rbar
    xbar_lcl = xbarbar - A2 * rbar
    r_ucl = D4 * rbar
    r_lcl = D3 * rbar

    xbar_ooc = (
        (summary["xbar"] > xbar_ucl)
        | (summary["xbar"] < xbar_lcl)
    )
    r_ooc = (
        (summary["range"] > r_ucl)
        | (summary["range"] < r_lcl)
    )

    summary["X-bar LCL"] = xbar_lcl
    summary["X-bar CL"] = xbarbar
    summary["X-bar UCL"] = xbar_ucl
    summary["R LCL"] = r_lcl
    summary["R CL"] = rbar
    summary["R UCL"] = r_ucl
    summary["X-bar 界外"] = xbar_ooc
    summary["R 界外"] = r_ooc

    return {
        "summary": summary,
        "xbarbar": xbarbar,
        "rbar": rbar,
        "xbar_ucl": xbar_ucl,
        "xbar_lcl": xbar_lcl,
        "r_ucl": r_ucl,
        "r_lcl": r_lcl,
        "xbar_ooc": summary.loc[xbar_ooc, "subgroup"].tolist(),
        "r_ooc": summary.loc[r_ooc, "subgroup"].tolist(),
        "stable": not (xbar_ooc.any() or r_ooc.any()),
    }


def make_control_plot(control):
    summary = control["summary"]
    x = summary["subgroup"]

    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

    axes[0].plot(x, summary["xbar"], marker="o")
    axes[0].axhline(control["xbarbar"], color="green", label="Center Line")
    axes[0].axhline(
        control["xbar_ucl"],
        color="red",
        linestyle="--",
        label="Control Limits"
    )
    axes[0].axhline(
        control["xbar_lcl"],
        color="red",
        linestyle="--"
    )
    axes[0].set_title("X-bar Chart")
    axes[0].set_ylabel("Subgroup Mean")
    axes[0].legend()
    axes[0].grid(alpha=0.25)

    axes[1].plot(x, summary["range"], marker="o")
    axes[1].axhline(control["rbar"], color="green", label="Center Line")
    axes[1].axhline(
        control["r_ucl"],
        color="red",
        linestyle="--",
        label="Control Limits"
    )
    axes[1].axhline(
        control["r_lcl"],
        color="red",
        linestyle="--"
    )
    axes[1].set_title("R Chart")
    axes[1].set_xlabel("Subgroup")
    axes[1].set_ylabel("Range")
    axes[1].legend()
    axes[1].grid(alpha=0.25)

    fig.tight_layout()
    return fig



def calculate_capability(process, lsl, usl, control):
    values = process["measurement"].to_numpy(dtype=float)
    mean = float(np.mean(values))
    sigma_overall = float(np.std(values, ddof=1))
    sigma_within = float(control["rbar"] / d2)

    if sigma_within <= 0 or sigma_overall <= 0:
        raise ValueError("能力分析的標準差必須大於 0。")

    cp = (usl - lsl) / (6 * sigma_within)
    cpu = (usl - mean) / (3 * sigma_within)
    cpl = (mean - lsl) / (3 * sigma_within)
    cpk = min(cpu, cpl)

    pp = (usl - lsl) / (6 * sigma_overall)
    ppu = (usl - mean) / (3 * sigma_overall)
    ppl = (mean - lsl) / (3 * sigma_overall)
    ppk = min(ppu, ppl)

    # 以整體平均與整體標準差的常態模型估計規格外機率
    ppm_below = norm.cdf(lsl, loc=mean, scale=sigma_overall) * 1_000_000
    ppm_above = (
        1 - norm.cdf(usl, loc=mean, scale=sigma_overall)
    ) * 1_000_000

    result = {
        "平均值": mean,
        "組內標準差估計": sigma_within,
        "整體標準差": sigma_overall,
        "Cp": cp,
        "Cpu": cpu,
        "Cpl": cpl,
        "Cpk": cpk,
        "Pp": pp,
        "Ppu": ppu,
        "Ppl": ppl,
        "Ppk": ppk,
        "常態模型預估低於 LSL (ppm)": ppm_below,
        "常態模型預估高於 USL (ppm)": ppm_above,
        "常態模型預估總不良率 (ppm)": ppm_below + ppm_above,
    }

    return pd.DataFrame(
        [{"指標": key, "結果": value} for key, value in result.items()]
    ), result


def make_capability_plot(process, capability, lsl, usl, target):
    values = process["measurement"].to_numpy(dtype=float)

    x = np.linspace(
        min(values.min(), lsl) - 0.02,
        max(values.max(), usl) + 0.02,
        400
    )

    fig, ax = plt.subplots(figsize=(8, 4.5))

    ax.hist(
        values,
        bins="auto",
        density=True,
        alpha=0.45,
        edgecolor="white",
        label="Observed Data"
    )

    ax.plot(
        x,
        norm.pdf(
            x,
            capability["平均值"],
            capability["整體標準差"]
        ),
        color="darkorange",
        linewidth=2,
        label="Fitted Normal Curve"
    )

    ax.axvline(lsl, color="red", linestyle="--", label="LSL")
    ax.axvline(usl, color="red", linestyle="--", label="USL")
    ax.axvline(target, color="green", linestyle=":", label="Target")

    ax.set_title("Process Capability")
    ax.set_xlabel("Measurement")
    ax.set_ylabel("Density")
    ax.grid(alpha=0.2)
    ax.legend()

    fig.tight_layout()
    return fig


# =========================================================
# Gradio 更新函式
# =========================================================

def reload_and_analyze(lsl, usl, target):
    """
    每次呼叫都重新讀取 CSV。
    因此替換資料檔後，按「更新資料」即可重跑分析。
    """
    try:
        lsl = float(lsl)
        usl = float(usl)
        target = float(target)

        if lsl >= usl:
            raise ValueError("規格錯誤：LSL 必須小於 USL。")

        check_csv_exists(MSA_FILE)
        check_csv_exists(PROCESS_FILE)

        # 按下按鈕時重新讀取，不使用舊的記憶體資料
        msa = pd.read_csv(MSA_FILE)
        process = pd.read_csv(PROCESS_FILE)

        warnings_list = []

        # 對原始檔缺少的欄位提醒，不自行補造資料
        for name, df in [("MSA", msa), ("製程", process)]:
            if "unit" not in df.columns:
                warnings_list.append(
                    f"{name} 資料沒有 unit 欄；請確認其單位與畫面設定一致。"
                )

        for col in ["timestamp", "machine_id", "batch_id"]:
            if col not in process.columns:
                warnings_list.append(
                    f"製程資料沒有 {col} 欄，無法依該欄位追溯或分層。"
                )

        # MSA
        msa_result = calculate_grr(msa, lsl, usl)
        msa_plot = make_msa_plot(msa)

        # 製程穩定性
        control = analyze_xbar_r(process)
        control_plot = make_control_plot(control)

        # 依 MSA 與穩定性設定分析閘門
        capability_df = pd.DataFrame()
        capability_plot = None
        capability_message = "能力分析未執行。"

        if not msa_result["msa_pass"]:
            capability_message = (
                "MSA 未通過本程式設定的教學門檻；"
                "不產生製程能力結論。"
            )
        elif not control["stable"]:
            capability_message = (
                "管制圖發現 3-sigma 界外點；"
                "建議先調查特殊原因，不發布能力結論。"
            )
        else:
            capability_df, capability = calculate_capability(
                process, lsl, usl, control
            )

            capability_plot = make_capability_plot(
                process, capability, lsl, usl, target
            )

            # 正態性檢定僅作診斷提示，不單獨決定分布方法
            values = process["measurement"].to_numpy(dtype=float)
            if len(values) >= 3:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    p_value = shapiro(values).pvalue

                if p_value < 0.05:
                    warnings_list.append(
                        "Shapiro-Wilk p < 0.05：常態假設需搭配直方圖、"
                        "機率圖及製程知識進一步審查。"
                    )

            capability_message = (
                "MSA 閘門通過，且本範例的管制圖界限檢查未發現界外點；"
                "已計算常態模型能力指標。"
            )

        # 摘要狀態文字
        status_lines = [
            f"**資料更新時間：** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}",
            f"**MSA 閘門：** {'通過' if msa_result['msa_pass'] else '未通過'} "
            f"（%Tolerance={msa_result['pct_tolerance']:.2f}%，"
            f"ndc={msa_result['ndc']}）",
            f"**穩定性檢查：** "
            f"{'未發現 3-sigma 界外點' if control['stable'] else '發現界外點'}",
            f"**能力分析：** {capability_message}",
            "",
            "**分析限制：** 本範例只檢查 3-sigma 界外點，"
            "尚未加入 Nelson／WECO 連續點規則；能力估計採常態模型。",
        ]

        if control["xbar_ooc"]:
            status_lines.append(
                f"- X-bar 圖界外子組：{control['xbar_ooc']}"
            )
        if control["r_ooc"]:
            status_lines.append(
                f"- R 圖界外子組：{control['r_ooc']}"
            )
        if warnings_list:
            status_lines.append("")
            status_lines.append("**資料提醒：**")
            status_lines.extend([f"- {item}" for item in warnings_list])

        status = "\n\n".join(status_lines)

        return (
            status,
            msa.head(30),
            process.head(50),
            msa_result["summary"],
            msa_result["anova"],
            control["summary"],
            capability_df,
            msa_plot,
            control_plot,
            capability_plot,
        )

    except Exception as exc:
        return empty_outputs(f"## 更新或分析失敗\n\n`{type(exc).__name__}: {exc}`")


# =========================================================
# 建立 Gradio 介面
# =========================================================

with gr.Blocks(title="SPC / MSA 製程能力分析") as demo:
    gr.Markdown(
        """
        # SPC / MSA 製程能力分析儀表板

        啟動時自動讀取程式所在資料夾的 MSA 與製程 CSV。
        若你替換了 CSV，請按「更新資料」重新載入並重算。
        """
    )

    with gr.Row():
        lsl_input = gr.Number(
            label="LSL",
            value=19.900,
            precision=6
        )
        usl_input = gr.Number(
            label="USL",
            value=20.100,
            precision=6
        )
        target_input = gr.Number(
            label="Target",
            value=20.000,
            precision=6
        )
        refresh_button = gr.Button(
            "更新資料",
            variant="primary"
        )

    status_box = gr.Markdown(label="分析狀態")

    with gr.Tab("分析摘要"):
        with gr.Row():
            msa_summary_table = gr.Dataframe(
                label="MSA 摘要",
                interactive=False
            )
            capability_table = gr.Dataframe(
                label="製程能力指標",
                interactive=False
            )

        subgroup_table = gr.Dataframe(
            label="X-bar / R 子組統計與管制界限",
            interactive=False
        )

    with gr.Tab("管制圖與能力圖"):
        control_plot_component = gr.Plot(label="X-bar / R ")
        capability_plot_component = gr.Plot(label="SPC scatter plat")

    with gr.Tab("MSA"):
        msa_plot_component = gr.Plot(label="average measurement")
        anova_table = gr.Dataframe(
            label="MSA ANOVA 表",
            interactive=False
        )

    with gr.Tab("原始資料"):
        msa_data_table = gr.Dataframe(
            label="MSA 原始資料（最多顯示前 30 列）",
            interactive=False
        )
        process_data_table = gr.Dataframe(
            label="製程原始資料（最多顯示前 50 列）",
            interactive=False
        )

    outputs = [
        status_box,
        msa_data_table,
        process_data_table,
        msa_summary_table,
        anova_table,
        subgroup_table,
        capability_table,
        msa_plot_component,
        control_plot_component,
        capability_plot_component,
    ]

    # 按鈕：重讀 CSV 並更新全部結果
    refresh_button.click(
        fn=reload_and_analyze,
        inputs=[lsl_input, usl_input, target_input],
        outputs=outputs,
    )

    # 啟動介面時自動讀取一次
    demo.load(
        fn=reload_and_analyze,
        inputs=[lsl_input, usl_input, target_input],
        outputs=outputs,
    )


if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
